# 📊 Notebook 4 — Evaluate with RAGAS

## RAGAS Evaluation Series · Part 4 of 4

This notebook loads `ragas_input.json` from Notebook 3 and evaluates:

- Faithfulness
- Answer Relevancy
- Context Precision
- Context Recall

Designed for:
- Python 3.12
- Ollama workflow
- RAGAS 0.2+


In [8]:
# Install (run once)

!pip install -U ragas datasets pandas


  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.2.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.16-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.15-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.14-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.13-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.1.11-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-google-vertexai 3.2.3 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.3.86 which is incompatible.
langchain-ollama 1.1.0 requires langchain-core<2.0.0,>=1.2.21, but you have langchain-core 0.3.86 which is incompatible.
langchain-tests 1.1.9 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-

In [9]:
! pip install langchain-community

In [10]:
!pip install ragas==0.2.15
!pip install langchain==0.3.26
!pip install langchain-community==0.3.27
!pip install langchain-core==0.3.74
!pip install langchain-ollama
!pip install datasets pandas

  Using cached ragas-0.2.15-py3-none-any.whl.metadata (9.0 kB)
Using cached ragas-0.2.15-py3-none-any.whl (190 kB)
  Attempting uninstall: ragas
    Found existing installation: ragas 0.4.3
    Uninstalling ragas-0.4.3:
      Successfully uninstalled ragas-0.4.3
  Using cached langchain_core-0.3.74-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain_core-0.3.74-py3-none-any.whl (443 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.3.74 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.74 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-google-vertexai 3.2.3 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.3.74 which is incompatible.
langchain-ollama 1.1.0 requires langchain-core<2.0.0,>=1.2.21, but you have langchain-core 0.3.74 which is incompatible.
langchain-openai 0.3.35 requires langchain-core<1.0.0,>=0.3.78, but you have langchain-core 0.3.74 which is incompatible.
langchain-tests 1.1.9 requires langchain-core<2.0.0,>=1.4.0, but you ha

  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
Using cached langchain_core-1.4.0-py3-none-any.whl (548 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.74
    Uninstalling langchain-core-0.3.74:
      Successfully uninstalled langchain-core-0.3.74


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.26 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.4.0 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-community 0.3.27 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.4.0 which is incompatible.
langchain-openai 0.3.35 requires langchain-core<1.0.0,>=0.3.78, but you have langchain-core 1.4.0 which is incompatible.


In [11]:
import json
import pandas as pd

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

print("✅ Imports successful")


✅ Imports successful


In [12]:
with open('ragas_input.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ Loaded {len(data)} records")
dataset = Dataset.from_list(data)
dataset


✅ Loaded 5 records


Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})

In [13]:
from ragas import evaluate

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


# Evaluator LLM
evaluator_llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

# Evaluator embeddings
evaluator_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# RAGAS wrappers
ragas_llm = LangchainLLMWrapper(evaluator_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(
    evaluator_embeddings
)

results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("✅ Evaluation complete")

Evaluating: 100%|██████████| 20/20 [01:30<00:00,  4.55s/it]


✅ Evaluation complete


In [14]:
df = results.to_pandas()

print(df.head())
df


                                        user_input  \
0              What is the maternity leave policy?   
1     How many days annual leave do employees get?   
2    What is the notice period before resignation?   
3                    Can employees work from home?   
4  What is the probation period for new employees?   

                                  retrieved_contexts  \
0  [Employees are eligible for 6 months maternity...   
1  [Employees are eligible for 6 months maternity...   
2  [Employees are eligible for 6 months maternity...   
3  [Employees are eligible for 6 months maternity...   
4  [Employees are eligible for 6 months maternity...   

                                            response  \
0  Employees are eligible for 6 months maternity ...   
1                                           24 days.   
2   The notice period before resignation is 60 days.   
3  Yes—employees are allowed to work from home tw...   
4  I could not find that information in the docum...   

 

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the maternity leave policy?,[Employees are eligible for 6 months maternity...,Employees are eligible for 6 months maternity ...,Employees are eligible for 6 months maternity ...,1.0,0.807232,1.0,1.0
1,How many days annual leave do employees get?,[Employees are eligible for 6 months maternity...,24 days.,Annual leave balance is 24 days.,1.0,0.562418,1.0,1.0
2,What is the notice period before resignation?,[Employees are eligible for 6 months maternity...,The notice period before resignation is 60 days.,Employees must serve 60 days notice period bef...,1.0,1.000000,1.0,1.0
3,Can employees work from home?,[Employees are eligible for 6 months maternity...,Yes—employees are allowed to work from home tw...,Work from home is allowed twice a week.,1.0,0.853614,1.0,1.0
4,What is the probation period for new employees?,[Employees are eligible for 6 months maternity...,I could not find that information in the docum...,Probation period for new employees is 3 months.,0.0,0.000000,0.0,0.0


In [15]:
print("\n📈 Aggregate Scores\n")

for metric in [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]:
    if metric in df.columns:
        print(f"{metric:20s}: {df[metric].mean():.4f}")



📈 Aggregate Scores

faithfulness        : 0.8000
answer_relevancy    : 0.6447
context_precision   : 0.8000
context_recall      : 0.8000
